In [22]:
import dash
from dash import dcc, html, dash_table, Input, Output
import pandas as pd
import plotly.express as px
import os

csv_file = "jobs.csv"
if os.path.exists(csv_file) and os.path.getsize(csv_file) > 0:
    df = pd.read_csv(csv_file)
else:
    df = pd.DataFrame([
        {"Job Title": "Data Analyst", "Company": "Vodafone", "Location": "Cairo, Egypt"},
        {"Job Title": "BI Developer", "Company": "Etisalat", "Location": "Giza, Egypt"}
    ])

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("📊 Dashboard سوق العمل", style={'textAlign': 'center'}),
    
    html.Label("اختر المكان:"),
    dcc.Dropdown(
        id='location-filter',
        options=[{'label': i, 'value': i} for i in df['Location'].dropna().unique()],
        placeholder="كل الأماكن",
        clearable=True
    ),
    html.Br(),
    
    html.Div([
        html.Div([html.H4("إجمالي الوظائف"), html.H2(id="kpi-jobs")], style={'border': '1px solid #ccc', 'padding': '10px', 'width': '22%', 'display': 'inline-block', 'textAlign': 'center'}),
        html.Div([html.H4("عدد الشركات"), html.H2(id="kpi-companies")], style={'border': '1px solid #ccc', 'padding': '10px', 'width': '22%', 'display': 'inline-block', 'textAlign': 'center', 'marginLeft': '2%'}),
        html.Div([html.H4("عدد المدن"), html.H2(id="kpi-locations")], style={'border': '1px solid #ccc', 'padding': '10px', 'width': '22%', 'display': 'inline-block', 'textAlign': 'center', 'marginLeft': '2%'}),
        html.Div([html.H4("أعلى شركة توظيفاً"), html.H2(id="kpi-top-company", style={'fontSize': '18px'})], style={'border': '1px solid #ccc', 'padding': '10px', 'width': '22%', 'display': 'inline-block', 'textAlign': 'center', 'marginLeft': '2%'}),
    ]),
    
    html.Br(),
    
    html.Div([
        dcc.Graph(id='chart-companies', style={'width': '49%', 'display': 'inline-block'}),
        dcc.Graph(id='chart-locations', style={'width': '49%', 'display': 'inline-block'}),
    ]),
    
    html.H3("جدول البيانات:"),
    dash_table.DataTable(id='data-table', page_size=5)
])

@app.callback(
    [Output('kpi-jobs', 'children'),
     Output('kpi-companies', 'children'),
     Output('kpi-locations', 'children'),
     Output('kpi-top-company', 'children'),
     Output('chart-companies', 'figure'),
     Output('chart-locations', 'figure'),
     Output('data-table', 'data')],
    [Input('location-filter', 'value')]
)
def update_dashboard(selected_location):
    filtered_df = df if not selected_location else df[df['Location'] == selected_location]
    
    total_jobs = len(filtered_df)
    total_companies = filtered_df['Company'].nunique() if not filtered_df.empty else 0
    total_locations = filtered_df['Location'].nunique() if not filtered_df.empty else 0
    top_company = filtered_df['Company'].mode()[0] if not filtered_df.empty and not filtered_df['Company'].empty else "N/A"
    
    comp_counts = filtered_df['Company'].value_counts().reset_index()
    comp_counts.columns = ['Company', 'count']
    fig1 = px.bar(comp_counts, x='Company', y='count', title="أكثر الشركات طلباً للوظائف")
    
    fig2 = px.pie(filtered_df, names='Location', title="توزيع الوظائف حسب المنطقة")
    
    return total_jobs, total_companies, total_locations, top_company, fig1, fig2, filtered_df.to_dict('records')

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=8050, debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
